In [1]:
import pandas as pd
from scipy import stats

jogadores = pd.read_csv("../data/processed/jogadores_saque.csv")

In [2]:
canhotos = jogadores[jogadores['hand'] == 'L']['pct_pontos_saque_ganhos']
destros = jogadores[jogadores['hand'] == 'R']['pct_pontos_saque_ganhos']

t_stat, p_valor = stats.ttest_ind(canhotos, destros, equal_var=False)
print(f"t = {t_stat:.3f}, p-valor = {p_valor:.4f}")

t = -5.461, p-valor = 0.0000


In [3]:
n1, n2 = len(canhotos), len(destros)
pooled_std = ((n1-1)*canhotos.std()**2 + (n2-1)*destros.std()**2) / (n1+n2-2)
pooled_std = pooled_std ** 0.5

cohens_d = (canhotos.mean() - destros.mean()) / pooled_std
print(f"Cohen's d = {cohens_d:.4f}")

Cohen's d = -0.0565


In [4]:
hard = jogadores[jogadores['surface'] == 'Hard']['pct_pontos_saque_ganhos']
clay = jogadores[jogadores['surface'] == 'Clay']['pct_pontos_saque_ganhos']
grass = jogadores[jogadores['surface'] == 'Grass']['pct_pontos_saque_ganhos']

f_stat, p_valor = stats.f_oneway(hard, clay, grass)
print(f"F = {f_stat:.3f}, p-valor = {p_valor:.4f}")

F = 872.220, p-valor = 0.0000


In [ ]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

dados_superficie = jogadores[jogadores['surface'].isin(['Hard', 'Clay', 'Grass'])]

tukey = pairwise_tukeyhsd(endog=dados_superficie['pct_pontos_saque_ganhos'], groups=dados_superficie['surface'], alpha=0.05)
print(tukey)

Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj  lower  upper  reject
--------------------------------------------------
  Clay  Grass   4.2092   0.0  3.9486 4.4698   True
  Clay   Hard   2.2784   0.0  2.1108 2.4459   True
 Grass   Hard  -1.9308   0.0 -2.1746 -1.687   True
--------------------------------------------------


In [6]:
import numpy as np

def cohens_d(grupo1, grupo2):
    n1, n2 = len(grupo1), len(grupo2)
    pooled_std = np.sqrt(((n1-1)*grupo1.std()**2 + (n2-1)*grupo2.std()**2) / (n1+n2-2))
    return (grupo1.mean() - grupo2.mean()) / pooled_std

print("Clay vs Grass:", cohens_d(clay, grass))
print("Clay vs Hard:", cohens_d(clay, hard))
print("Grass vs Hard:", cohens_d(grass, hard))

Clay vs Grass: -0.47931192979043297
Clay vs Hard: -0.2552504107983749
Grass vs Hard: 0.21833027648962586


In [8]:
for nome, grupo in [('Hard', hard), ('Clay', clay), ('Grass', grass)]:
    ic = stats.t.interval(0.95, len(grupo)-1, loc=grupo.mean(), scale=stats.sem(grupo))
    print(f"{nome}: média={grupo.mean():.2f}, IC 95% = ({ic[0]:.2f}, {ic[1]:.2f})")

Hard: média=64.13, IC 95% = (64.04, 64.21)
Clay: média=61.85, IC 95% = (61.73, 61.96)
Grass: média=66.06, IC 95% = (65.88, 66.23)


In [9]:
dados_ht = jogadores.dropna(subset=['ht'])
r_ht, p_ht = stats.pearsonr(dados_ht['ht'], dados_ht['pct_pontos_saque_ganhos'])
print(f"Altura: r = {r_ht:.4f}, p-valor = {p_ht:.4f}")

dados_age = jogadores.dropna(subset=['age'])
r_age, p_age = stats.pearsonr(dados_age['age'], dados_age['pct_pontos_saque_ganhos'])
print(f"Idade: r = {r_age:.4f}, p-valor = {p_age:.4f}")

Altura: r = 0.2054, p-valor = 0.0000
Idade: r = 0.0120, p-valor = 0.0009
